# 엑셀 파일을 분석하기 위해 필요한 라이브러리

In [ ]:
!pip install openpyxl

### 한글을 plot에 표시하기 위해 필요한 사전 작업

In [ ]:
import platform 
import matplotlib.pyplot as plt 

# Mac인 경우 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우용
else:
    plt.rc('font', family='DejaVu Sans')    # 리눅스 기본
# 마이너스 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd

# 엑셀 읽기 (2줄 헤더)
df = pd.read_excel("출생률.xlsx", engine="openpyxl", header=[0, 1])

df.head(len(df))


In [ ]:
import numpy as np

df.replace('-', np.nan, inplace=True)

# '출생아수'에 해당하는 열만 선택
cols_to_check = [col for col in df.columns if col[1] == '출생아수']

# 이 열들 중 하나라도 NaN이 있는 column(연도)만 제거
cols_with_nan = [col for col in cols_to_check if df[col].isna().any()]

# 해당 열과 함께 매칭되는 '합계출산율'도 같이 삭제
related_cols = []
for col in cols_with_nan:
    year = col[0]
    related_cols.append((year, '출생아수'))
    related_cols.append((year, '합계출산율'))

# 제거
df_cleaned = df.drop(columns=related_cols)
df_cleaned.head(len(df_cleaned))

In [ ]:
# 기존 컬럼 복사
new_columns = []

for year, item in df_cleaned.columns:
    # 연도 부분 수정
    if '2024' in year:
        new_columns.append(('2024', item))
    else:
        new_columns.append((year, item))

# 새로운 컬럼 설정
df_cleaned.columns = pd.MultiIndex.from_tuples(new_columns)
df_cleaned.head(len(df_cleaned))

In [ ]:
df_cleaned.info()

In [ ]:
# 1. 행정구역 컬럼 추출
region_col = [col for col in df_cleaned.columns if '시군구별' in col[0]][0]

# 2. "합계출산율" 열만 추출
birth_cols = [col for col in df_cleaned.columns if '합계출산율' in col[1]]

# 3. 필요한 열만 추출
df_birth = df_cleaned[[region_col] + birth_cols].copy()

# 4. 열 이름 단순화: 첫 열은 '지역', 나머지는 연도 (col[0])만
df_birth.columns = ['지역'] + [col[0] for col in birth_cols]

# 5. 확인
df_birth.head()

In [ ]:
df_birth.info()

In [ ]:

year = '2024'

# 1. 특정년 데이터 정렬 (전국 제외)
df_year = df_birth[df_birth['지역'] != '전국'][['지역', year]].copy()
df_year = df_year.sort_values(year, ascending=False) # 내림차순 정렬 (높은 값 → 낮은 값)

# 2. 전국 평균값 따로 추출
national_avg = df_birth[df_birth['지역'] == '전국'][year].values[0]

# 3. 시각화
plt.figure(figsize=(12, 6))
bars = plt.bar(df_year['지역'], df_year[year], color='steelblue')

# 4. 평균선 추가
plt.axhline(y=national_avg, color='red', linestyle='--', label=f'전국 평균 ({national_avg:.3f})')

# 5. 기타 설정
plt.title(f'{year}년 시도별 합계출산율 (전국 평균선 포함)')
plt.ylabel('합계출산율')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
seoul = df_birth[df_birth['지역'] == '서울특별시'].iloc[0, 1:] # Pandas의 Series 로 구성됨. 이에 따라 index, value 형태로 출력.
jeonnam = df_birth[df_birth['지역'] == '전라남도'].iloc[0, 1:]


# 열 이름을 정수형으로 변환
seoul.index = seoul.index.astype(int)
seoul = seoul.astype(float)  # 값도 수치형으로 변환
jeonnam.index = jeonnam.index.astype(int)
jeonnam = jeonnam.astype(float)

서울특별시 지역의 연도별 출산율과 전라남도의 연도별 출산율 비교 그래프

In [ ]:
# x축 연도
years = seoul.index

# 시각화
plt.figure(figsize=(12, 6))
plt.plot(years, seoul, label='서울특별시', color='blue', marker='o')
plt.plot(years, jeonnam, label='전라남도', color='green', marker='s')

plt.title('서울특별시 vs 전라남도 연도별 합계출산율')
plt.xlabel('연도')
plt.ylabel('합계출산율')
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 출산율과 보육시설과의 상관관계 분석

In [ ]:
# 엑셀 읽기 (2줄 헤더)
factor1 = pd.read_excel("보육시설.xlsx", engine="openpyxl", header=[0, 1])
factor1.head()

In [ ]:
factor1.info()

In [ ]:
# 1. 행정구역 컬럼 추출
region_col = [col for col in factor1.columns if '행정구역별' in col[0]][0]

# 2. "유아 천명당 보육시설수" 열만 추출
facility_cols = [col for col in factor1.columns if '유아 천명당 보육시설수' in col[1]]

# 3. 필요한 열만 추출
df_facility = factor1[[region_col] + facility_cols].copy()

# 4. 열 이름 단순화: 첫 열은 '지역', 나머지는 연도 (col[0])만
df_facility.columns = ['지역'] + [col[0] for col in facility_cols]

# 5. 확인
df_facility.head(len(df_facility))

In [ ]:
# 1. 2024년 데이터 정렬 (전국 제외)
df_2024 = df_facility[df_facility['지역'] != '전국'][['지역', '2024']].copy()
df_2024 = df_2024.sort_values('2024', ascending=False) # 내림차순 정렬 (높은 값 → 낮은 값)

# 2. 전국 평균값 따로 추출
national_avg = df_facility[df_facility['지역'] == '전국']['2024'].values[0]

# 3. 시각화
plt.figure(figsize=(12, 6))
bars = plt.bar(df_2024['지역'], df_2024['2024'], color='steelblue')

# 4. 평균선 추가
plt.axhline(y=national_avg, color='red', linestyle='--', label=f'전국 평균 ({national_avg:.3f})')

# 5. 기타 설정
plt.title('2024년 시도별 유아 1000명당 보육시설  (전국 평균선 포함)')
plt.ylabel('유아 1000명당 보육시설')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
seoul = df_facility[df_facility['지역'] == '서울특별시'].iloc[0, 1:] # Pandas의 Series 로 구성됨. 이에 따라 index, value 형태로 출력.
jeonnam = df_facility[df_facility['지역'] == '전라남도'].iloc[0, 1:]


# 열 이름을 정수형으로 변환
seoul.index = seoul.index.astype(int)
seoul = seoul.astype(float)  # 값도 수치형으로 변환
jeonnam.index = jeonnam.index.astype(int)
jeonnam = jeonnam.astype(float)

# x축 연도
years = seoul.index

# 시각화
plt.figure(figsize=(12, 6))
plt.plot(years, seoul, label='서울특별시', color='blue', marker='o')
plt.plot(years, jeonnam, label='전라남도', color='green', marker='s')

plt.title('서울특별시 vs 전라남도 연도별 2024년 시도별 유아 1000명당 보육시설')
plt.xlabel('연도')
plt.ylabel('2024년 시도별 유아 1000명당 보육시설')
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. 공통 연도 추출
common_years = list(set(df_birth.columns[1:]) & set(df_facility.columns[1:]))
common_years = sorted(common_years)

# 2. long-format으로 변환
df_birth_long = df_birth.melt(id_vars='지역', value_vars=common_years,
                              var_name='연도', value_name='출산율')

df_facility_long = df_facility.melt(id_vars='지역', value_vars=common_years,
                                    var_name='연도', value_name='보육')

# 3. merge
df_merged = pd.merge(df_birth_long, df_facility_long, on=['지역', '연도'])

# 4. pivot: (행=지역, 열=(연도, 변수))
df_pivoted = df_merged.pivot(index='지역', columns='연도', values=['출산율', '보육'])

# 5. 열 순서 정리 (출산율-보육 순서로 정렬되도록 전치 후 다시 전치)
df_pivoted = df_pivoted.swaplevel(axis=1).sort_index(axis=1)

# 6. 확인
df_pivoted.head(len(df_pivoted))


In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns

# 1. 전체 연도에 대해 출산율 vs 보육 데이터 수집
data = []

for year in df_pivoted.columns.levels[0]:
    if (year, '출산율') in df_pivoted.columns and (year, '보육') in df_pivoted.columns:
        birth = df_pivoted[(year, '출산율')]
        housing = df_pivoted[(year, '보육')]

        for region, b, h in zip(df_pivoted.index, birth, housing):
            if pd.notna(b) and pd.notna(h):
                data.append({'지역': region, '연도': int(year), '출산율': b, '보육': h})

# 2. DataFrame 생성
df_all = pd.DataFrame(data)

# 3. 전체 상관계수 계산
r = df_all['출산율'].corr(df_all['보육'])

# 4. 시각화 (회귀선 포함)
plt.figure(figsize=(8, 6))
sns.regplot(data=df_all, x='출산율', y='보육', scatter_kws={'alpha': 0.7}, line_kws={'color': 'gray', 'linestyle': 'dashed'})
plt.title(f"출산율 vs 보육 (전체 연도 통합)\n피어슨 상관계수 r = {r:.2f}", fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# 1. 최근 6년 추출
recent_years = sorted(df_all['연도'].unique())[-6:]
n_rows, n_cols = 2, 3

# 2. 플롯 준비
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

# 3. 각 연도별 플롯
for i, year in enumerate(recent_years):
    ax = axes[i]
    subset = df_all[df_all['연도'] == year]

    # 산점도
    sns.scatterplot(data=subset, x='출산율', y='보육', ax=ax, color='teal')

    # 회귀선 (optional)
    sns.regplot(data=subset, x='출산율', y='보육', ax=ax, scatter=False, color='gray', line_kws={'linestyle': 'dashed'})

    # 상관계수 계산
    corr = subset[['출산율', '보육']].corr(method='pearson').iloc[0, 1]

    # 타이틀
    ax.set_title(f"{year} (r = {corr:.2f})", fontsize=11)
    ax.grid(True)

# 4. 공통 레이블
fig.suptitle("최근 6년 출산율 vs 보육시설수 (연도별)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# 출산율과 주거 안정성 (자가 비율 + 무료 )과의 상관관계 분석

In [ ]:
# 엑셀 읽기 (2줄 헤더)
factor2 = pd.read_excel("주거실태_점유형태.xlsx", engine="openpyxl", header=[0, 1])
factor2.head() 

In [ ]:
# 1. 행정구역 컬럼 추출
region_col = [col for col in factor2.columns if '시도구분(1)' in col[0]][0]

# 2. "1- 임차 = 자가+ 무상" 열만 추출
housing_cols = [col for col in factor2.columns if '임차' in col[1]]

# 3. 필요한 열만 추출
df_housing= factor2[[region_col] +  housing_cols].copy()

# 4. 임차 → 자가 비율로 변환: 값 변경
for col in housing_cols:
    df_housing[col] = 100 - df_housing[col]

# 5. 열 이름 단순화: '지역' + 연도만
df_housing.columns = ['지역'] + [col[0] for col in housing_cols]

# 6. 확인
df_housing.head()

In [ ]:
# 1. 2023년 데이터 정렬 (전국 제외)
df_2023 = df_housing[df_facility['지역'] != '전국'][['지역', '2023']].copy()
df_2023 = df_2023.sort_values('2023', ascending=False) # 내림차순 정렬 (높은 값 → 낮은 값)

# 2. 전국 평균값 따로 추출
national_avg = df_housing[df_housing['지역'] == '전국']['2023'].values[0]

# 3. 시각화
plt.figure(figsize=(12, 6))
bars = plt.bar(df_2023['지역'], df_2023['2023'], color='steelblue')

# 4. 평균선 추가
plt.axhline(y=national_avg, color='red', linestyle='--', label=f'전국 평균 ({national_avg:.3f})')

# 5. 기타 설정
plt.title('2023년 시도별 주거 안정률  (전국 평균선 포함)')
plt.ylabel('주거 안정률')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 1. full name 리스트 (기준)
full_names = df_birth['지역'].unique().tolist()  # 예: df1에서

# 2. short name 리스트 (변환 대상)
short_names = df_housing['지역'].unique().tolist()  # 예: df2에서

# 3. 자동 매핑 생성
region_map = {}

for short in short_names:
    candidates = []

    for full in full_names:
        # 조건 1: 짧은 이름이 긴 이름에 포함되거나
        # 조건 2: 짧은 이름의 각 글자가 긴 이름에 모두 들어 있음
        if short in full or all(char in full for char in short):
            candidates.append(full)

    if len(candidates) == 1:
        region_map[short] = candidates[0]
    else:
        print(f"⚠️ '{short}' 매핑 모호하거나 없음 → 후보: {candidates}")

In [ ]:
region_map

In [ ]:
df_housing['지역'] = df_housing['지역'].map(region_map)
df_housing.head()

In [ ]:
# 1. 공통 연도 추출
common_years = list(set(df_birth.columns[1:]) & set(df_housing.columns[1:]))
common_years = sorted(common_years)

# 2. long-format으로 변환
df_birth_long = df_birth.melt(id_vars='지역', value_vars=common_years,
                              var_name='연도', value_name='출산율')

df_housing_long = df_housing.melt(id_vars='지역', value_vars=common_years,
                                    var_name='연도', value_name='주거안정율')

# 3. merge
df_merged = pd.merge(df_birth_long, df_housing_long, on=['지역', '연도'])

# 4. pivot: (행=지역, 열=(연도, 변수))
df_pivoted = df_merged.pivot(index='지역', columns='연도', values=['출산율', '주거안정율'])

# 5. 열 순서 정리 (출산율-보육 순서로 정렬되도록 전치 후 다시 전치)
df_pivoted = df_pivoted.swaplevel(axis=1).sort_index(axis=1)

# 6. 확인
df_pivoted.head(len(df_pivoted))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 전체 연도에 대해 출산율 vs 주거안정율 데이터 수집
data = []

for year in df_pivoted.columns.levels[0]:
    if (year, '출산율') in df_pivoted.columns and (year, '주거안정율') in df_pivoted.columns:
        birth = df_pivoted[(year, '출산율')]
        housing = df_pivoted[(year, '주거안정율')]

        for region, b, h in zip(df_pivoted.index, birth, housing):
            if pd.notna(b) and pd.notna(h):
                data.append({'지역': region, '연도': int(year), '출산율': b, '주거안정율': h})

# 2. DataFrame 생성
df_all = pd.DataFrame(data)

# 3. 전체 상관계수 계산
r = df_all['출산율'].corr(df_all['주거안정율'])

# 4. 시각화 (회귀선 포함)
plt.figure(figsize=(8, 6))
sns.regplot(data=df_all, x='출산율', y='주거안정율', scatter_kws={'alpha': 0.7}, line_kws={'color': 'gray', 'linestyle': 'dashed'})
plt.title(f"출산율 vs 주거안정율 (전체 연도 통합)\n피어슨 상관계수 r = {r:.2f}", fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 최근 3개 연도 선택 (또는 원하는 연도)
years = sorted(df_pivoted.columns.levels[0])[-3:]  # 예: [2021, 2022, 2023]

# 2. subplot 준비
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

# 3. 연도별 산점도
for i, year in enumerate(years):
    ax = axes[i]
    x = df_pivoted[(year, '출산율')]
    y = df_pivoted[(year, '주거안정율')]

    # 산점도 + 회귀선
    sns.regplot(x=x, y=y, ax=ax, scatter_kws={'alpha': 0.7}, line_kws={'linestyle': 'dashed', 'color': 'gray'})

    # 상관계수
    r = x.corr(y)
    ax.set_title(f"{year}년 (r = {r:.2f})")
    ax.set_xlabel('출산율')
    if i == 0:
        ax.set_ylabel('주거안정율')
    else:
        ax.set_ylabel("")

    ax.grid(True)

# 4. 전체 제목
fig.suptitle("출산율 vs 주거안정율 (연도별 산점도)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


# 출산율과 삶의 만족도 간의 상관관계 분석

In [ ]:
# 엑셀 읽기 (2줄 헤더)
factor3 = pd.read_excel("만족도.xlsx", engine="openpyxl", header=[0, 1])
factor3.head() 

In [ ]:
# 1. 행정구역 컬럼 지정
region_col = [col for col in factor3.columns if '행정구역별' in col[0]][0]

# 2. 결측값 채움 (NaN → 위의 행정구역 이름으로 채우기), 위에서 아래로" 순서대로 진행
factor3[region_col] = factor3[region_col].ffill()

# 2. 남자 / 여자 데이터 필터링
male_rows = factor3[ factor3[('특성별(2)', '특성별(2)')] == '남자' ]
female_rows = factor3[ factor3[('특성별(2)', '특성별(2)')] == '여자']

# 3. "매우 만족", "약간 만족" 컬럼만 추출
satisfaction_cols = [col for col in factor3.columns
                     if ('매우 만족' in str(col[1]) or '약간 만족' in str(col[1]))
                     and col[0].isdigit()] # 연도 부분이 숫자형 문자열인지 확인

df_male = male_rows[[region_col] + satisfaction_cols].copy() # 독립적인 객체 생성
df_male = df_male.set_index(region_col)

df_female = female_rows[[region_col] + satisfaction_cols].copy() # 독립적인 객체 생성
df_female = df_female.set_index(region_col)

# 3. 연도별 합산 ("매우 만족", "약간 만족" )
df_life_male = pd.DataFrame(index=df_male.index)
df_life_female = pd.DataFrame(index=df_female.index)

# 4. 연도별 합산
years = [col[0] for col in satisfaction_cols ]

for year in years:
    cols = [(year, '매우 만족'), (year, '약간 만족')]
    if all(col in df_male.columns for col in cols):
        df_life_male[year] = df_male[cols].sum(axis=1)
        df_life_female[year] = df_female[cols].sum(axis=1)

df_life_male = df_life_male.reset_index(drop=False)  # '행정구역'을 열로 복원
df_life_female = df_life_female.reset_index(drop=False)  # '행정구역'을 열로 복원

# 5. 열 이름 단순화: '지역' + 연도만
df_life_male.columns = ['지역'] + df_life_male.columns.tolist()[1:]
df_life_female.columns = ['지역'] + df_life_female.columns.tolist()[1:]

In [ ]:
df_life_male.head()

In [ ]:
# 1. '전국' 제외한 2024년 데이터 추출
male_2024 = df_life_male[df_life_male['지역'] != '전국'][['지역', '2024']].copy()
female_2024 = df_life_female[df_life_female['지역'] != '전국'][['지역', '2024']].copy()

# 2. 정렬 (같은 순서 보장)
male_2024 = male_2024.sort_values('지역')
female_2024 = female_2024.sort_values('지역')

# 3. 전국 평균 계산 (제외된 '전국'이 아니라 지역들의 평균)
male_avg = male_2024['2024'].mean()
female_avg = female_2024['2024'].mean()

# 4. 시각화
plt.figure(figsize=(12, 6))
plt.plot(male_2024['지역'], male_2024['2024'], marker='o', label='남성', color='blue')
plt.plot(female_2024['지역'], female_2024['2024'], marker='o', label='여성', color='red')

# 5. 평균선 표시
plt.axhline(y=male_avg, color='blue', linestyle='--', alpha=0.6, label=f'남성 평균 ({male_avg:.1f})')
plt.axhline(y=female_avg, color='red', linestyle='--', alpha=0.6, label=f'여성 평균 ({female_avg:.1f})')

# 6. 커스터마이징
plt.xticks(rotation=45, ha='right')
plt.ylabel('삶의 만족도 (매우+약간 만족 %)')
plt.title('2024년 지역별 남성 vs 여성 삶의 만족도')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. 도시 지정
city1 = '서울특별시'
city2 = '울산광역시'

# 2. 연도 리스트
years = df_life_male.columns[1:].astype(int)

# 3. 남성 데이터
male_city1 = df_life_male[df_life_male['지역'] == city1].iloc[0, 1:]
male_city2 = df_life_male[df_life_male['지역'] == city2].iloc[0, 1:]

# 4. 여성 데이터
female_city1 = df_life_female[df_life_female['지역'] == city1].iloc[0, 1:]
female_city2 = df_life_female[df_life_female['지역'] == city2].iloc[0, 1:]

# 5. 그림 그리기
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# 왼쪽: 남성
axes[0].plot(years, male_city1, marker='o', label=city1, color='red')
axes[0].plot(years, male_city2, marker='o', label=city2, color='blue')
axes[0].set_title('남성 삶의 만족도')
axes[0].set_xlabel('연도')
axes[0].set_ylabel('만족도 (매우+약간 %)')
axes[0].grid(True)
axes[0].legend()

# 오른쪽: 여성
axes[1].plot(years, female_city1, marker='o', label=city1, color='red')
axes[1].plot(years, female_city2, marker='o', label=city2, color='blue')
axes[1].set_title('여성 삶의 만족도')
axes[1].set_xlabel('연도')
axes[1].grid(True)
axes[1].legend()

# 전체 제목 + 정리
fig.suptitle(f'{city1} vs {city2} 남녀 삶의 만족도 (연도별)', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
# 1. long-format 변환
male_long = df_life_male.melt(id_vars='지역', var_name='연도', value_name='남성만족도')
female_long = df_life_female.melt(id_vars='지역', var_name='연도', value_name='여성만족도')
birth_long = df_birth.melt(id_vars='지역', var_name='연도', value_name='출산율')

# 2. 연도 숫자형으로 통일
for df in [male_long, female_long, birth_long]:
    df['연도'] = df['연도'].astype(int)

# 3. 병합
merged = birth_long.merge(male_long, on=['지역', '연도'])
merged = merged.merge(female_long, on=['지역', '연도'])

# 4. pivoted 구조로 변환
df_pivoted = merged.pivot(index='지역', columns='연도', values=['출산율', '남성만족도', '여성만족도'])

# 5. 열 정렬 (출산율 → 남성 → 여성 순으로)
df_pivoted = df_pivoted.swaplevel(axis=1).sort_index(axis=1)

df_pivoted.head(len(df_pivoted))

In [ ]:
df_long = df_pivoted.stack(level=0).rename_axis(['지역', '연도']).reset_index()

# 2. subplot 생성 (1행 2열)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

# 3. 남성 만족도 vs 출산율
r_male = df_long['출산율'].corr(df_long['남성만족도'])
sns.regplot(data=df_long, x='출산율', y='남성만족도', ax=axes[0], color='blue', scatter_kws={'alpha': 0.7})
axes[0].set_title(f'출산율 vs 남성 만족도\n피어슨 r = {r_male:.2f}')
axes[0].set_xlabel('출산율')
axes[0].set_ylabel('남성 만족도')
axes[0].grid(True)

# 4. 여성 만족도 vs 출산율
r_female = df_long['출산율'].corr(df_long['여성만족도'])
sns.regplot(data=df_long, x='출산율', y='여성만족도', ax=axes[1], color='red', scatter_kws={'alpha': 0.7})
axes[1].set_title(f'출산율 vs 여성 만족도\n피어슨 r = {r_female:.2f}')
axes[1].set_xlabel('출산율')
axes[1].set_ylabel('여성 만족도')
axes[1].grid(True)

# 5. 전체 레이아웃
fig.suptitle('출산율과 성별 만족도의 상관관계 (전체 연도 통합)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 최근 6년 추출
recent_years = sorted(df_long['연도'].unique())[-6:]
n_cols = 3
n_rows = 2

# 2. 남성 만족도 vs 출산율
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, year in enumerate(recent_years):
    ax = axes[i]
    subset = df_long[df_long['연도'] == year]
    r = subset['출산율'].corr(subset['남성만족도'])
    sns.regplot(data=subset, x='출산율', y='남성만족도', ax=ax, color='blue', scatter_kws={'alpha':0.7})
    ax.set_title(f"{year} (r = {r:.2f})")
    ax.grid(True)

    # 왼쪽 열일 때만 y축 라벨
    if i % n_cols == 0:
        ax.set_ylabel('남성만족도')
    else:
        ax.set_ylabel('')

    # 아래쪽 행일 때만 x축 라벨
    if i >= n_cols * (n_rows - 1):
        ax.set_xlabel('출산율')
    else:
        ax.set_xlabel('')

fig.suptitle("출산율 vs 남성 만족도 (최근 6년)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


# 3. 여성 만족도 vs 출산율
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, year in enumerate(recent_years):
    ax = axes[i]
    subset = df_long[df_long['연도'] == year]
    r = subset['출산율'].corr(subset['여성만족도'])
    sns.regplot(data=subset, x='출산율', y='여성만족도', ax=ax, color='red', scatter_kws={'alpha':0.7})
    ax.set_title(f"{year} (r = {r:.2f})")
    ax.grid(True)

    if i % n_cols == 0:
        ax.set_ylabel('여성만족도')
    else:
        ax.set_ylabel('')

    if i >= n_cols * (n_rows - 1):
        ax.set_xlabel('출산율')
    else:
        ax.set_xlabel('')

fig.suptitle("출산율 vs 여성 만족도 (최근 6년)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


# 출산율과 주관적 소득 간의 상관관계 분석

In [ ]:
# 엑셀 읽기 (2줄 헤더)
factor4 = pd.read_excel("주관적소득.xlsx", engine="openpyxl", header=[0, 1])
factor4.head() 

In [ ]:
# 1. 행정구역 컬럼 지정
region_col = [col for col in factor4.columns if '행정구역별' in col[0]][0]

# 2. 결측값 채움 (NaN → 위의 행정구역 이름으로 채우기), 위에서 아래로" 순서대로 진행
factor4[region_col] = factor4[region_col].ffill()

# 3. '--'  열 제거
drop_cols = [col for col in factor4.columns if factor4[col].astype(str).eq('-').any() ]
factor4 = factor4.drop(columns=drop_cols)


# 2. 남자 / 여자 데이터 필터링
male_rows = factor4[ factor4[('특성별(2)', '특성별(2)')] == '남자' ]
female_rows = factor4[ factor4[('특성별(2)', '특성별(2)')] == '여자']

# 3. 소득부족 = 약간 부족+ 매우 부족 컬럼만 추출
satisfaction_cols = [col for col in factor4.columns
                     if ('약간 부족함' in str(col[1]) or '매우 부족함' in str(col[1]))
                     and col[0].isdigit()] # 연도 부분이 숫자형 문자열인지 확인

df_male = male_rows[[region_col] + satisfaction_cols].copy() # 독립적인 객체 생성
df_male = df_male.set_index(region_col)

df_female = female_rows[[region_col] + satisfaction_cols].copy() # 독립적인 객체 생성
df_female = df_female.set_index(region_col)

# 3. 소득부족 = 약간 부족+ 매우 부족 컬럼
df_earn_male = pd.DataFrame(index=df_male.index)
df_earn_female = pd.DataFrame(index=df_female.index)


# 4. 연도별 합산
years = [col[0] for col in satisfaction_cols ]

for year in years:
    cols = [(year, '약간 부족함'), (year, '매우 부족함')]
    if all(col in df_male.columns for col in cols):
        df_earn_male[year] = df_male[cols].sum(axis=1)
        df_earn_female[year] = df_female[cols].sum(axis=1)

df_earn_male = df_earn_male.reset_index(drop=False)  # '행정구역'을 열로 복원
df_earn_female = df_earn_female.reset_index(drop=False)  # '행정구역'을 열로 복원

# 5. 열 이름 단순화: '지역' + 연도만
df_earn_male.columns = ['지역'] + df_earn_male.columns.tolist()[1:]
df_earn_female.columns = ['지역'] + df_earn_female.columns.tolist()[1:]

In [ ]:
df_earn_female.head(len(df_earn_female))

In [ ]:
year = '2015'

# 1. '전국' 제외한 특정 년도 데이터 추출
male= df_earn_male[df_earn_male['지역'] != '전국'][['지역', year]].copy()
female= df_earn_female[df_earn_female['지역'] != '전국'][['지역', year]].copy()

# 2. 정렬 (같은 순서 보장)
male= male.sort_values('지역')
female= female.sort_values('지역')

# 3. 전국 평균 계산 (제외된 '전국'이 아니라 지역들의 평균)
male_avg = male[year].mean()
female_avg = female[year].mean()

# 4. 시각화
plt.figure(figsize=(12, 6))
plt.plot(male['지역'], male[year], marker='o', label='남성', color='blue')
plt.plot(female['지역'], female[year], marker='o', label='여성', color='red')

# 5. 평균선 표시
plt.axhline(y=male_avg, color='blue', linestyle='--', alpha=0.6, label=f'남성 평균 ({male_avg:.1f})')
plt.axhline(y=female_avg, color='red', linestyle='--', alpha=0.6, label=f'여성 평균 ({female_avg:.1f})')

# 6. 커스터마이징
plt.xticks(rotation=45, ha='right')
plt.ylabel('주관적 빈곤도 (약간 부족+ 매우 부족 %)')
plt.title(f'{year}년 지역별 남성 vs 여성의 주관적 빈곤도')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. 도시 지정
city1 = '서울특별시'
city2 = '울산광역시'

# 2. 연도 리스트
years = df_earn_male.columns[1:].astype(int)

# 3. 남성 데이터
male_city1 = df_earn_male[df_life_male['지역'] == city1].iloc[0, 1:]
male_city2 = df_earn_male[df_life_male['지역'] == city2].iloc[0, 1:]

# 4. 여성 데이터
female_city1 = df_earn_female[df_life_female['지역'] == city1].iloc[0, 1:]
female_city2 = df_earn_female[df_life_female['지역'] == city2].iloc[0, 1:]

# 5. 그림 그리기
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# 왼쪽: 남성
axes[0].plot(years, male_city1, marker='o', label=city1, color='red')
axes[0].plot(years, male_city2, marker='o', label=city2, color='blue')
axes[0].set_title('남성 소득 주관적 빈곤도')
axes[0].set_xlabel('연도')
axes[0].set_ylabel('빈곤도 (매우+약간 %)')
axes[0].grid(True)
axes[0].legend()

# 오른쪽: 여성
axes[1].plot(years, female_city1, marker='o', label=city1, color='red')
axes[1].plot(years, female_city2, marker='o', label=city2, color='blue')
axes[1].set_title('여성 소득 주관적 빈곤도')
axes[1].set_xlabel('연도')
axes[1].set_ylim(40,80)
axes[1].grid(True)
axes[1].legend()

# 전체 제목 + 정리
fig.suptitle(f'{city1} vs {city2} 남녀 소득 주관적 빈곤도 (연도별)', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
# 1. long-format 변환
male_long = df_earn_male.melt(id_vars='지역', var_name='연도', value_name='남성 주관적 빈곤도')
female_long = df_earn_female.melt(id_vars='지역', var_name='연도', value_name='여성 주관적 빈곤도')
birth_long = df_birth.melt(id_vars='지역', var_name='연도', value_name='출산율')

# 2. 연도 숫자형으로 통일
for df in [male_long, female_long, birth_long]:
    df['연도'] = df['연도'].astype(int)

# 3. 병합
merged = birth_long.merge(male_long, on=['지역', '연도'])
merged = merged.merge(female_long, on=['지역', '연도'])

# 4. pivoted 구조로 변환
df_pivoted = merged.pivot(index='지역', columns='연도', values=['출산율', '남성 주관적 빈곤도', '여성 주관적 빈곤도'])

# 5. 열 정렬 (출산율 → 남성 → 여성 순으로)
df_pivoted = df_pivoted.swaplevel(axis=1).sort_index(axis=1)

df_pivoted.head(len(df_pivoted))

In [ ]:
df_long = df_pivoted.stack(level=0).rename_axis(['지역', '연도']).reset_index()

# 1. 최근 5년 추출
recent_years = sorted(df_long['연도'].unique())[-5:]
n_cols = 3
n_rows = 2

# 2. 남성 만족도 vs 출산율
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, year in enumerate(recent_years):
    ax = axes[i]
    subset = df_long[df_long['연도'] == year]
    r = subset['출산율'].corr(subset['남성 주관적 빈곤도'])
    sns.regplot(data=subset, x='출산율', y='남성 주관적 빈곤도', ax=ax, color='blue', scatter_kws={'alpha':0.7})
    ax.set_title(f"{year} (r = {r:.2f})")
    ax.grid(True)

    # 왼쪽 열일 때만 y축 라벨
    if i % n_cols == 0:
        ax.set_ylabel('남성 주관적 빈곤도')
    else:
        ax.set_ylabel('')

    # 아래쪽 행일 때만 x축 라벨
    if i >= n_cols * (n_rows - 1):
        ax.set_xlabel('출산율')
    else:
        ax.set_xlabel('')

fig.suptitle("출산율 vs 남성 주관적 빈곤도 (최근 6년)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


# 3. 여성 만족도 vs 출산율
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, year in enumerate(recent_years):
    ax = axes[i]
    subset = df_long[df_long['연도'] == year]
    r = subset['출산율'].corr(subset['여성 주관적 빈곤도'])
    sns.regplot(data=subset, x='출산율', y='여성 주관적 빈곤도', ax=ax, color='red', scatter_kws={'alpha':0.7})
    ax.set_title(f"{year} (r = {r:.2f})")
    ax.grid(True)

    if i % n_cols == 0:
        ax.set_ylabel('여성 주관적 빈곤도')
    else:
        ax.set_ylabel('')

    if i >= n_cols * (n_rows - 1):
        ax.set_xlabel('출산율')
    else:
        ax.set_xlabel('')

fig.suptitle("출산율 vs 여성 주관적 빈곤도 (최근 6년)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
df_long = df_pivoted.stack(level=0).rename_axis(['지역', '연도']).reset_index()

# 2. subplot 생성 (1행 2열)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

# 3. 남성 만족도 vs 출산율
r_male = df_long['출산율'].corr(df_long['남성 주관적 빈곤도'])
sns.regplot(data=df_long, x='출산율', y='남성 주관적 빈곤도', ax=axes[0], color='blue', scatter_kws={'alpha': 0.7})
axes[0].set_title(f'출산율 vs 남성 주관적 빈곤도\n피어슨 r = {r_male:.2f}')
axes[0].set_xlabel('출산율')
axes[0].set_ylabel('남성 만족도')
axes[0].grid(True)

# 4. 여성 만족도 vs 출산율
r_female = df_long['출산율'].corr(df_long['여성 주관적 빈곤도'])
sns.regplot(data=df_long, x='출산율', y='여성 주관적 빈곤도', ax=axes[1], color='red', scatter_kws={'alpha': 0.7})
axes[1].set_title(f'출산율 vs 여성 주관적 빈곤도\n피어슨 r = {r_female:.2f}')
axes[1].set_xlabel('출산율')
axes[1].set_ylabel('여성 만족도')
axes[1].grid(True)

# 5. 전체 레이아웃
fig.suptitle('출산율과 성별 주관적 빈곤도의 상관관계 (전체 연도 통합)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# 모든 요소 고려

In [ ]:
from functools import reduce

# 1. 분석에 사용할 변수명과 데이터프레임 매핑
data_sources = {
    '출산율': df_birth,
    '보육': df_facility,
    '주거안정율': df_housing,
    '남성만족도': df_life_male,
    '여성만족도': df_life_female,
    '남성빈곤도': df_earn_male,
    '여성빈곤도': df_earn_female
}

# 2. 공통 연도 추출
common_years = set(df_birth.columns[1:])  # 첫 번째는 '지역'
for df in data_sources.values():
    common_years &= set(df.columns[1:])
common_years = sorted(map(int, common_years))  # 연도 정렬

# 3. 각 데이터프레임을 long-format으로 변환
long_dfs = []
for var_name, df in data_sources.items():
    df_long = df[['지역'] + [str(y) for y in common_years]].melt(id_vars='지역', var_name='연도', value_name=var_name)
    df_long['연도'] = df_long['연도'].astype(int)
    long_dfs.append(df_long)

# 4. 순차 병합 (지역 + 연도 기준)
df_merged = reduce(lambda left, right: pd.merge(left, right, on=['지역', '연도']), long_dfs)

# 5. pivoted 구조로 변환: 행=지역, 열=(연도, 변수명)
df_pivoted = df_merged.pivot(index='지역', columns='연도')

# 6. 열 순서 정리: (연도, 변수명) → (변수명, 연도)
df_pivoted = df_pivoted.swaplevel(axis=1).sort_index(axis=1)
df_pivoted.columns.name = None  # 열 이름 계층 제거

df_pivoted.head()

In [ ]:
df_pivoted.columns

In [ ]:
# 1. 연도 목록 추출
years = sorted(df_pivoted.columns.levels[0])
n_cols = 3
n_rows = (len(years) + n_cols - 1) // n_cols

# 2. subplot 준비
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
axes = axes.flatten()

# 3. 연도별 상관행렬 히트맵
for i, year in enumerate(years):
    ax = axes[i]
    try:
        # 연도에 해당하는 데이터 추출
        df_year = df_pivoted[year]
        # 상관계수 계산
        corr = df_year.corr()
        # 히트맵 그리기
        sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, ax=ax, cbar=False)
        ax.set_title(f'{year}년 상관계수 행렬')
    except Exception as e:
        ax.set_title(f'{year}년: 오류 발생')
        print(f'⚠️ {year}년 처리 중 오류: {e}')
        ax.axis('off')

# 4. 빈 subplot 제거
for j in range(i+1, len(axes)):
    axes[j].axis('off')

# 5. 전체 제목 및 정리
plt.suptitle('연도별 변수 상관행렬', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. 변수 선택 (출산율 포함)
variables = ['출산율', '보육', '남성만족도', '여성만족도', '남성빈곤도', '여성빈곤도', '주거안정율']

# 2. 데이터 준비 (특정 연도 예: 2021)

year = 2021
df_year = df_pivoted[year].dropna()

X = df_year[variables]

# 3. 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 5. 변수 화살표 방향 (PC1, PC2 구성)
vectors = pca.components_.T  # 변수 방향 (len(variables), 2)

# 6. Biplot 시각화
plt.figure(figsize=(10, 8))

# ● 지역 점
for i, name in enumerate(X.index):
    plt.scatter(X_pca[i, 0], X_pca[i, 1], color='gray', alpha=0.6)
    plt.text(X_pca[i, 0]+0.05, X_pca[i, 1], name, fontsize=8)

# → 변수 벡터 화살표

scaled_factor = 2

for i, var in enumerate(variables):
    plt.arrow(0, 0, vectors[i, 0]*scaled_factor, vectors[i, 1]*scaled_factor,
              color='red' if var == '출산율' else 'black', width=0.005, head_width=0.05)
    plt.text(vectors[i, 0]*scaled_factor*1.1, vectors[i, 1]*scaled_factor*1.1, var,
             color='red' if var == '출산율' else 'black', fontsize=10)

# 꾸미기
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title(f"PCA Biplot (출산율 포함) - {year}년")
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

# PCA 결과
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=variables).round(3)
print(loadings)

In [ ]:
# 2. 데이터 준비 (특정 연도 예: 2023)
year = 2023
df_year = df_pivoted[year].dropna()

X = df_year[variables]

# 3. 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 5. 변수 화살표 방향 (PC1, PC2 구성)
vectors = pca.components_.T  # 변수 방향 (len(variables), 2)

# 6. Biplot 시각화
plt.figure(figsize=(10, 8))

# ● 지역 점
for i, name in enumerate(X.index):
    plt.scatter(X_pca[i, 0], X_pca[i, 1], color='gray', alpha=0.6)
    plt.text(X_pca[i, 0]+0.05, X_pca[i, 1], name, fontsize=8)

# → 변수 벡터 화살표

scaled_factor = 2.5

for i, var in enumerate(variables):
    plt.arrow(0, 0, vectors[i, 0]*scaled_factor, vectors[i, 1]*scaled_factor,
              color='red' if var == '출산율' else 'black', width=0.005, head_width=0.05)
    plt.text(vectors[i, 0]*scaled_factor*1.1, vectors[i, 1]*scaled_factor*1.1, var,
             color='red' if var == '출산율' else 'black', fontsize=10)

# 꾸미기
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title(f"PCA Biplot (출산율 포함) - {year}년")
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

# PCA 결과
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=variables).round(3)
print(loadings)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# 1. 변수 리스트 (출산율 포함)
variables = ['출산율', '보육', '남성만족도', '여성만족도', '남성빈곤도', '여성빈곤도', '주거안정율']

# 2. PCA 로딩 벡터 (PC1, PC2만 고려)
loadings = pd.DataFrame(pca.components_.T, index=variables, columns=['PC1', 'PC2'])

# 3. cosine similarity 계산
target = '출산율'
cos_sims = {}

for var in variables:
    if var != target:
        cos_sim = cosine_similarity([loadings.loc[target]], [loadings.loc[var]])[0][0]
        cos_sims[var] = cos_sim

# 4. 결과 정리
cosine_df = pd.Series(cos_sims).sort_values(ascending=False).round(3)
print("📐 출산율과 각 변수 벡터 간 cosine similarity:")
print(cosine_df)

# PCR 기반 회귀 분석

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

# 1. 데이터 준비
variables = ['보육', '남성만족도', '여성만족도', '남성빈곤도', '여성빈곤도', '주거안정율']
year = 2023
df_year = df_pivoted[year].dropna()

X = df_year[variables]
y = df_year['출산율']

# 2. X 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. PCA 적용
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# 4. 누적 분산으로 주성분 수 결정 (예: 80% 이상)
explained_ratio = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(explained_ratio >= 0.8) + 1
print(f"선택된 주성분 개수: {n_components}")

# 5. 회귀분석 (PCR)
X_pca_reduced = X_pca[:, :n_components]
model = LinearRegression()
model.fit(X_pca_reduced, y)

# 6. 결과 확인
y_pred = model.predict(X_pca_reduced)
r_squared = model.score(X_pca_reduced, y)

print(f"R² (설명력): {r_squared:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y, y_pred, color='blue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('실제 출산율')
plt.ylabel('예측 출산율 (PCR)')
plt.title('PCR 기반 출산율 예측')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 7. 회귀계수 변환: 주성분 → 원래 변수
V = pca.components_[:n_components, :].T  # shape: (p, n_components)
beta_pcr = model.coef_  # shape: (n_components,)
beta_orig = V @ beta_pcr  # 환산된 회귀계수 (원래 변수 기준)

# 8. 정리 및 출력
coef_series = pd.Series(beta_orig, index=variables, name='β (원 변수 기준)').sort_values(ascending=False).round(4)
print("\n📌 원래 변수 기준 PCR 회귀계수:")
print(coef_series)

# 기타 데이터

- 맞벌이 가구 : url = 'https://www.dropbox.com/scl/fi/r1ngtjfsmvrcpwolbtkla/.xlsx?rlkey=hxmzitn4giuu5bi5ttov6bffu&dl=1'

- 물가지수: url = 'https://www.dropbox.com/scl/fi/lrd3i4ojx3cbucn467hyu/.xlsx?rlkey=matxe2pvemuwjq63v1odnewnc&dl=1'

- 범죄율 : url = 'https://www.dropbox.com/scl/fi/fo5isdnww2nfmkxx35z8i/.xlsx?rlkey=485g9hw5xcpwirpkbt8y2a3y9&dl=1'

- 청년 고용률 : url = 'https://www.dropbox.com/scl/fi/u6lg536b4l610qrgz258h/.xlsx?rlkey=jf39xpg4cvf1t56nz1ljkjq6d&dl=1'


- 어린이집 현황 : url ='https://www.dropbox.com/scl/fi/kw7cl1mtp7agjajgurfco/.xlsx?rlkey=dfhh14uya3hkyjp9v36yymydt&dl=1'

In [ ]:
# 맞벌이 가구
factor = pd.read_excel("맞벌이가구.xlsx", engine="openpyxl", header=[0, 1])
factor.head()

In [ ]:
# 물가지수
factor = pd.read_excel("물가지수.xlsx", engine="openpyxl")
factor.head(20)

In [ ]:
# 범죄율
factor = pd.read_excel("범죄율.xlsx", engine="openpyxl", header=[0, 1])
factor.head()

In [ ]:
#청년 고용률
factor = pd.read_excel("청년고용률.xlsx", engine="openpyxl" )
factor.head()

In [ ]:
#어린이집 현황
factor = pd.read_excel("어린이집현황.xlsx", engine="openpyxl" )
factor.head(7)